# OrbitLab: CUDA N-body gravity
Select a GPU runtime. Upload **orbitlab-source.zip** from the deliverables. Each cell invokes the repository; lessons and equations are in README/docs. Temporary Colab files disappear on runtime replacement.

In [ ]:
from pathlib import Path
import subprocess,os,json
for command in (["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],["nvcc","--version"]):
    print(subprocess.check_output(command,text=True))


## Acquire saved sources
Upload the supplied source archive. After authorized GitHub publication, this cell can be replaced with a clone of the verified URL.

In [ ]:
from google.colab import files
import zipfile,io
uploaded=files.upload()
assert "orbitlab-source.zip" in uploaded,"Choose orbitlab-source.zip"
root=Path("/content/orbitlab-clean")
assert not root.exists(),"Use a fresh runtime or a new empty directory"
with zipfile.ZipFile(io.BytesIO(uploaded["orbitlab-source.zip"])) as archive:
    for item in archive.infolist():
        target=(root/item.filename).resolve()
        assert target.is_relative_to(root.resolve()),"Unsafe archive member"
    archive.extractall(root)
assert (root/"CMakeLists.txt").exists()


In [ ]:
def run(command):
    result=subprocess.run(list(map(str,command)),cwd=root,env=os.environ,capture_output=True,text=True)
    print(result.stdout+result.stderr)
    result.check_returncode()
    return result
run(["python","-m","pip","install","-r","requirements.txt"])
cap=subprocess.check_output(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],text=True).splitlines()[0].strip().replace(".","")
run(["cmake","-S",".","-B","build-gpu","-DCMAKE_BUILD_TYPE=Release","-DORBITLAB_ENABLE_CUDA=ON",f"-DCMAKE_CUDA_ARCHITECTURES={cap}"])
run(["cmake","--build","build-gpu","--parallel","2"])
exe=root/"build-gpu/orbitlab"
os.environ["ORBITLAB_EXE"]=str(exe)


In [ ]:
run(["ctest","--test-dir","build-gpu","--output-on-failure"])
run(["python","-m","unittest","discover","-s","scripts","-p","test_*.py"])


In [ ]:
run(["python","scripts/run_science.py","--exe",exe,"--output","results/science"])
run(["python","scripts/plot_science.py","--input","results/science"])
run(["python","scripts/animate.py","--trajectory","results/science/cuda-tiled-float/trajectory.csv","--output","results/orbit.gif"])


## Performance experiment
This sweeps 120 configurations, including partial tiles and precision choices. On the tested T4 runtime this takes minutes. Timing variability is expected; inspect batch ranges, not just one ratio.

In [ ]:
run(["python","scripts/run_benchmarks.py","--exe",exe,"--output","results/benchmarks"])
run(["python","scripts/plot_benchmarks.py","--input","results/benchmarks"])
run(["python","scripts/profile_kernels.py","--exe",exe,"--output","results/profiling"])


In [ ]:
from IPython.display import display,Image
print(json.loads((root/"results/science/summary.json").read_text()))
display(Image(filename=str(root/"results/orbit.gif")))
display(Image(filename=str(root/"results/science/convergence.png")))


## Keep the results
Download before leaving Colab. The notebook saves no fabricated execution outputs; local verified logs/results accompany the project.

In [ ]:
import shutil
archive=shutil.make_archive("/content/orbitlab-run-results","zip",root,"results")
files.download(archive)
